In [161]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Constants for the function F for nitrous oxide in moist air at a total pressure of 1 atm (mol L-1 atm-1; Weiss & Price, 1980)
A1 = -165.8806
A2 = 222.8743
A3 = 92.0792
A4 = -1.48425
B1 = -0.056235
B2 = 0.031619
B3 = -0.0048472

# Constants for Schmidt number (Sc; Wanninkhof, 2014)
A_Sc = 2356.2
B_Sc = -166.38
C_Sc = 6.3952
D_Sc = -0.13422
E_Sc = 0.0011506

# Nitrous oxide atmospheric mole fraction at the time of sampling (parts per billion)
N2O_MOLFRAC = 331

# Wind speed at sampling coordinate (m s-1)
WIND_SPEED = 8.7

dataset = pd.read_csv("GP15_GP17_CP.csv") # Load your desired dataset
depth = []
temp = []
sal = []
n2o_obs = []

for i, row in dataset.iterrows():
    if row["station"] == "19_15": # Enter desired station number based on dataset
            depth.append(row["Depth"])
            temp.append(row["conservative_temp"])
            sal.append(row["absolute_salinity"])
            n2o_obs.append(row["n2o"])
            
for i in range(len(temp) - 1):
    temp_diff = abs(temp[i + 1] - temp[i])
    if temp_diff > 0.2:
        mixed_layer = depth[i]
        ml_index = i + 1
        break

temp_ml = sum(temp[:ml_index])/len(temp[:ml_index]) + 273.15 # unit: Kelvin
sal_ml = sum(sal[:ml_index])/len(sal[:ml_index])
n2o_ml = sum(n2o_obs[:ml_index])/len(n2o_obs[:ml_index])

F_function = np.exp(A1 + (A2*(100/temp_ml)) + A3*(np.log(temp_ml/100)) + A4*((temp_ml/100)**2) + 
                    sal_ml*(B1 + B2*(temp_ml/100) + B3*((temp_ml/100)**2)))

n2o_eq_0 = (N2O_MOLFRAC/1000000000) * F_function # unit: mol L-1
n2o_eq = n2o_eq_0 * 1000000000 # unit: nmol L-1
dn2o = n2o_ml - n2o_eq
n2o_sat = (n2o_ml/n2o_eq) * 100 # unit: %

wind_squared = WIND_SPEED ** 2 # unit: (m s-1)^2
temp_ml_c = round(temp_ml - 273.15, 2) # unit: Celsius
Sc = A_Sc + (B_Sc*temp_ml_c) + (C_Sc*(temp_ml_c ** 2)) + (D_Sc*(temp_ml_c ** 3)) + (E_Sc*(temp_ml_c ** 4))
k_cmhr = 0.251 * wind_squared * (1/np.sqrt(Sc/660)) # unit: cm hr-1
k_md = k_cmhr * 0.01 * 24 # unit: m d-1

# NOTE: nmol L-1 = µmol m-3
n2o_flux_mol = k_md * dn2o # unit: m d-1 µmol m-3 --> µmol m-2 d-1 
n2o_flux_mass = n2o_flux_mol * 44.013 # unit: Nitrous oxide mass: 44.013 gram mol-1 --> µgram m-2 d-1

# Uncertainties
dF_dT = (-A2*100/temp_ml**2 + A3/temp_ml + 2*A4*temp_ml/100**2 + 
          sal_ml*(B2/100 + 2*B3*temp_ml/100**2))
dF_dS = B1 + B2*(temp_ml/100) + B3*(temp_ml/100)**2
sigma_T = 0.005
sigma_S = 0.005
sigma_F = F_function * np.sqrt((dF_dT * sigma_T)**2 + (dF_dS * sigma_S)**2)
sigma_WIND = 0.20 * WIND_SPEED  # 20% relative uncertainty
sigma_Sc = 5
sigma_k_cmhr = k_cmhr * np.sqrt((2*sigma_WIND/WIND_SPEED)**2 + (0.5*sigma_Sc/Sc)**2)
sigma_k_md = sigma_k_cmhr * 0.01 * 24
n2o_ml_stdev = np.std(n2o_obs[:ml_index])
n2o_eq_stdev = sigma_F * N2O_MOLFRAC
sigma_dn2o = np.sqrt(n2o_ml_stdev**2 + n2o_eq_stdev**2)
sigma_flux = np.sqrt((sigma_k_md/k_md)**2 + (sigma_F/F_function)**2 + (sigma_dn2o/dn2o)**2) * abs(n2o_flux_mol)

print(f"Mixed Layer sample boundary = {mixed_layer:.2f} m")
print(f"Nitrous Oxide concentration in mixed layer = {n2o_ml:.2f} ± {n2o_ml_stdev:.2f} nmol L-1")
print(f"Nitrous Oxide equilibrium concentration in mixed layer = {n2o_eq:.2f} ± {n2o_eq_stdev:.2f} nmol L-1")
print(f"Total Nitrous Oxide added = {dn2o:.2f} ± {sigma_dn2o:.2f} nmol L-1")
print(f"Sea-air Nitrous Oxide flux = {n2o_flux_mol:.2f} ± {sigma_flux:.2f} µmol m-2 day-1")

Mixed Layer sample boundary = 25.00 m
Nitrous Oxide concentration in mixed layer = 6.51 ± 0.02 nmol L-1
Nitrous Oxide equilibrium concentration in mixed layer = 6.11 ± 0.00 nmol L-1
Total Nitrous Oxide added = 0.39 ± 0.03 nmol L-1
Sea-air Nitrous Oxide flux = 2.11 ± 0.86 µmol m-2 day-1
